In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df=pd.read_pickle(f"./df.pkl")
y=df[['dm']]
X=df.drop(columns=['dm'])

In [23]:
X.shape

(13796, 28)

In [24]:
X_trainvalid, X_test, y_trainvalid, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42,
            shuffle=True
        )

In [25]:
X_trainvalid.shape

(11036, 28)

In [26]:
X_train,X_valid, y_train, y_valid=train_test_split(X_trainvalid, y_trainvalid, test_size=0.2, random_state= 65, shuffle=True)

In [27]:
X_valid.shape, y_valid.shape

((2208, 28), (2208, 1))

In [28]:
from model.progression_scoring import progression_scoring
from model.optimize_state import compute_total_decision

import itertools
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

epsilons = [1,3,5,10]
lambdas = [0.001,0.005,0.01,0.05]

results=[]

X_eval = X_valid.sample(100, random_state=42)

BASE_DIR = Path.cwd()

model_paths = [
            str(BASE_DIR / "model" / "final_checkpoints" / f"fold_{i}" / "best-checkpoint-v4.ckpt")
            for i in range(5)
        ]

with open(BASE_DIR / "scalers.pkl", "rb") as f:
    scalers = pickle.load(f)
    
def apply_deltas(X, deltas):
        X_opt = X.copy()

        for col, delta in deltas.items():
            if col in X_opt.columns:
                X_opt[col] = X_opt[col] + delta

        return X_opt


for eps, lam in itertools.product(epsilons,lambdas):

    score_reduction=[]
    l1_changes=[]
    n_changes=[]

    for idx in range(len(X_eval)):

        x = X_eval.iloc[[idx]]

        before = progression_scoring(
            x,
            model_paths,
            scalers
        )

        deltas = compute_total_decision(
            x,
            model_paths,
            scalers,
            epsilon=eps,
            lambda_reg=lam
        )

        x_after = apply_deltas(x,deltas)

        after = progression_scoring(
            x_after,
            model_paths,
            scalers
        )

        score_reduction.append(before-after)

        l1_changes.append(
            np.sum(np.abs(list(deltas.values())))
        )

        n_changes.append(len(deltas))

    results.append({

        "epsilon":eps,
        "lambda":lam,

        "score_reduction":
        np.mean(score_reduction),

        "L1_change":
        np.mean(l1_changes),

        "n_modified":
        np.mean(n_changes)

    })

results=pd.DataFrame(results)

c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:56: The loaded checkpoint was produced with Lightning v2.5.2, which is newer than your current Lightning version: v2.5.1.post0
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightning\pytorch\utilities\migration\utils.py:56: The loaded checkpoint was produced with Lightning v2.5.2, which is newer than your current Lightning version: v2.5.1.post0
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightning\p

KeyboardInterrupt: 